# Run the baselines on Colab (T4, Gemma-2-2B-IT)

**One-time setup**
1. Accept the Gemma licence at https://huggingface.co/google/gemma-2-2b-it with your HF account.
2. In Colab: 🔑 *Secrets* (left sidebar) → add `HF_TOKEN` (a read token) and enable notebook access. The token is read into the process environment only; it is never printed or written to disk by this repo.
3. *Runtime → Change runtime type → T4 GPU.*

**Every session** (including after a disconnect): run cells 1-4, then continue from the first phase cell that has not finished.
Every phase writes to Google Drive, skips work that is already complete and resumes mid-way, so re-running a cell is always safe.
Expect roughly a day of T4 time in total, spread over several sessions; most of it is Baseline B's 100-random-direction steering nulls (see `docs/DECISIONS.md` D15 and the README).

In [ ]:
# 1. Drive + paths. Results live on Drive (survive runtime resets); assembled
#    activation arrays and the HF model cache stay on the fast local disk.
from google.colab import drive
drive.mount('/content/drive')
import os
RESULTS = '/content/drive/MyDrive/rfd/results'          # primary model: Gemma-2-2B-IT
os.makedirs(RESULTS, exist_ok=True)
os.environ['RFD_RESULTS_DIR'] = RESULTS
os.environ['RFD_SCRATCH_DIR'] = '/content/rfd_scratch'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('results ->', RESULTS)

In [ ]:
# 2. HF token from Colab secrets -> environment (never printed). Checks Gemma access.
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
from huggingface_hub import hf_hub_download
hf_hub_download('google/gemma-2-2b-it', 'config.json')
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')), '| Gemma-2 access OK')

In [ ]:
# 3. Get or update the code (local disk), then install pinned deps (keeping Colab's CUDA torch).
REPO = '/content/Reasoning-Failure-Diagnosis'
if os.path.isdir(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone https://github.com/mehul24d/Reasoning-Failure-Diagnosis.git {REPO}
%cd {REPO}
!git log --oneline -1
!grep -v '^torch==' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt && pip install -q -e . --no-deps
import torch, transformers
print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))

In [ ]:
# 4. Smoke test (a few minutes): unit tests, then the GPU checks: model loads,
#    pure-greedy generation config, one BOS, system text folded into the first
#    user turn, worst-case batches fit, activations fit float16, zero-strength
#    steering changes nothing. Stop here if anything FAILs.
!make test-fast
!make smoke

## Phases, in order
Each cell can be re-run after a disconnect; finished stages print `[resume] ... already complete`.

In [ ]:
# Phase 0: TransformerLens vs HF equivalence, padding, greedy reproducibility, generation config.
!make phase0
!cat {RESULTS}/phase0/equivalence.json | head -60

In [ ]:
# Phase 1: Baseline A (Sun et al.): error pilot, collection, probes, selective re-prompting.
!make baseline-a

In [ ]:
# Traces for Baseline B and the pilot: generation, step labels, activations.
!make traces
!cat {RESULTS}/traces/parse_report.json

In [ ]:
# Phase 2: Baseline B (Yuan et al.): failure probes, then steering with 100 random directions.
# The longest phase; it caches every evaluated condition, so just re-run after a disconnect.
!make baseline-b

In [ ]:
# Phase 3: pilot (readability, RQ3, repair), then the optional T=0.7 stability samples.
!make pilot
!make pilot-sample

In [ ]:
# Phase 4: regenerate the report tables (also copied to Drive next to the results).
!make report
!head -40 docs/BASELINE_REPORT_TABLES.md

In [ ]:
# Bring the results back: a zip of the measurement files (no activation shards or caches).
!cd {RESULTS}/.. && zip -qr /content/rfd_results.zip results -x '*/shards/*' '*/cache/*' '*.npy' '*.npz' && ls -lh /content/rfd_results.zip
from google.colab import files
files.download('/content/rfd_results.zip')

### Secondary model (optional, after the primary run)
Qwen2.5-1.5B-Instruct goes into its own folder; a results folder refuses a second model.
```python
os.environ['RFD_RESULTS_DIR'] = '/content/drive/MyDrive/rfd/results_qwen2_5_1_5b'
!make all HARNESS=configs/qwen2_5_1_5b.yaml
```